In [ ]:
import requests
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
API_KEY = os.getenv("NVIDIA_API_KEY")

# NVIDIA OpenAI-compatible endpoint
API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"

# LLM QUERY FUNCTION
def query_llm(prompt: str):

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "meta/llama3-70b-instruct",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.2
    }

    response = requests.post(API_URL, headers=headers, json=payload)
    data = response.json()

    try:
        return data["choices"][0]["message"]["content"]
    except:
        return str(data)


# tools
def flag_suspicious(transaction):
    return "Suspicious Transaction"

def mark_normal(transaction):
    return "Normal Transaction"


# AGENT
def fraud_agent(transaction: str):

    # Step 1: Thinking (Reasoning)
    thought = query_llm(f"Analyze this transaction: {transaction}")
    print("LLM Thought:", thought)

    # Step 2: Planning (Decision)
    if "overseas" in transaction.lower() or "multiple deposits" in transaction.lower():
        action = flag_suspicious
    else:
        action = mark_normal

    # Step 3: Acting
    result = action(transaction)

    # Step 4: Answering
    final_answer = query_llm(f"Final decision: {result}")
    return final_answer


# RUN
print(fraud_agent("Payment to overseas account"))